# BRAID v2 — quick evals: score saved models from Drive

Standalone companion to `braidv2_clip_mlp_colab copy.ipynb` — loads already-trained `model.pth`
checkpoints straight from Drive and re-runs the embedding + image-reconstruction evals against them.
No training happens here. Useful for re-scoring a past sweep, or scoring one model without re-running
the whole training notebook.

Assumes the same data layout as the training notebook:
- `visualroi_betas/subj01_visualroi_session{NN}.pt`  -> `[750, D]` (masked)
- `clip_embeds_subj01/subj01_clip_embeds{NN}.pt`      -> `[750, 1280]`
- Saved runs live under `/content/drive/MyDrive/braid2/runs/mlp_{loss_type}_{timestamp}/model.pth`


In [ ]:
import os, glob, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp -r /content/drive/MyDrive/braid2/nsd_bundle/visualroi_betas .

In [ ]:
!cp -r /content/drive/MyDrive/braid2/nsd_bundle/clip_embeds_subj01 .

In [ ]:
!cp -r /content/drive/MyDrive/braid2/nsd_stimuli.hdf5 .

In [ ]:
# --- config ---
BASE      = "/content/"
BETA_DIR  = f"{BASE}/visualroi_betas"
CLIP_DIR  = f"{BASE}/clip_embeds_subj01"
SUBJECT   = "subj01"
BATCH     = 512
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

device: cuda


## Load + pair data (shared-1000 protocol)
Identical split logic to the training notebook (same seed, same shared-1000/val/train partition), so
this reproduces the exact test set and normalization stats each model was actually trained and
evaluated against — necessary for the loaded checkpoints to be scored correctly.


In [ ]:
import numpy as np, urllib.request
from collections import defaultdict
from scipy.io import loadmat

def session_id(path):
    return int(''.join(c for c in os.path.basename(path)[-6:] if c.isdigit()))

beta_sess = {session_id(p): p for p in glob.glob(f"{BETA_DIR}/{SUBJECT}_visualroi_session*.pt")}
clip_sess = {session_id(p): p for p in glob.glob(f"{CLIP_DIR}/{SUBJECT}_clip_embeds*.pt")}
sessions  = sorted(set(beta_sess) & set(clip_sess))
print(f"{len(sessions)} paired sessions:", sessions)

betas = torch.cat([torch.load(beta_sess[s]).float() for s in sessions])   # [N, D]
clips = torch.cat([torch.load(clip_sess[s]).float() for s in sessions])   # [N, 1280]
print("betas:", tuple(betas.shape), "| clips:", tuple(clips.shape))

# --- NSD design: trial -> imgBrick image id, and the shared-1000 test images ---
EXP = f"{BASE}/nsd_expdesign.mat"
if not os.path.exists(EXP):
    urllib.request.urlretrieve(
        "https://natural-scenes-dataset.s3.amazonaws.com/nsddata/experiments/nsd/nsd_expdesign.mat", EXP)
mat            = loadmat(EXP)
masterordering = mat["masterordering"].reshape(-1).astype(np.int64) - 1    # [30000] subject-image idx
subjectim      = mat["subjectim"].astype(np.int64) - 1                     # [8,10000] -> imgBrick idx
imgbrick_ids   = subjectim[int(SUBJECT[-2:]) - 1, masterordering]          # [30000] imgBrick idx per trial
shared_ids     = set(mat["sharedix"].reshape(-1).astype(np.int64) - 1)     # 1000 shared imgBrick ids

def img_id(g):                                     # concat-trial idx -> imgBrick image id
    sess, trial = sessions[g // 750], g % 750
    return int(imgbrick_ids[(sess - 1) * 750 + trial])

img_of = np.array([img_id(g) for g in range(len(betas))])                  # image id per trial

# --- image-level split: shared-1000 -> test ; 5% of remaining images -> val ; rest -> train ---
is_test  = np.array([i in shared_ids for i in img_of])
rest_img = np.array(sorted(set(img_of[~is_test])))
np.random.RandomState(0).shuffle(rest_img)
val_img   = set(rest_img[:int(0.05 * len(rest_img))])
in_val    = np.array([i in val_img for i in img_of])
train_idx = np.where(~is_test & ~in_val)[0]
val_idx   = np.where(~is_test &  in_val)[0]
test_idx  = np.where(is_test)[0]
print(f"train {len(train_idx)} | val {len(val_idx)} | test trials {len(test_idx)} "
      f"| unique test images {len(set(img_of[test_idx]))}")

# --- normalization stats from TRAIN trials only (no leak into val/test) ---
beta_mean, beta_std = betas[train_idx].mean(0), betas[train_idx].std(0) + 1e-6
clip_mean, clip_std = clips[train_idx].mean(0), clips[train_idx].std(0) + 1e-6
torch.save({'mean': beta_mean, 'std': beta_std}, f"{BASE}/beta_norm_stats.pt")
torch.save({'mean': clip_mean, 'std': clip_std}, f"{BASE}/clip_norm_stats.pt")
def norm(b, c): return (b - beta_mean) / beta_std, (c - clip_mean) / clip_std

# --- train / val use all repeats (safe: shared images are fully held out) ---
train_ds = TensorDataset(*norm(betas[train_idx], clips[train_idx]))
val_ds   = TensorDataset(*norm(betas[val_idx],   clips[val_idx]))

# --- test: average each shared image's 3 repeats -> one denoised sample per image ---
groups = defaultdict(list)
for g in test_idx: groups[int(img_of[g])].append(g)
test_img_ids = list(groups.keys())
test_betas   = torch.stack([betas[gs].mean(0) for gs in groups.values()])  # raw, repeat-averaged
test_clips   = torch.stack([clips[gs[0]]       for gs in groups.values()])  # identical target per image
test_ds      = TensorDataset(*norm(test_betas, test_clips))

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH)
test_loader  = DataLoader(test_ds,  batch_size=BATCH)

40 paired sessions: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40]
betas: (30000, 4657) | clips: (30000, 1280)
train 25650 | val 1350 | test trials 3000 | unique test images 1000


## Model
Same architecture as the training notebook — needed to reconstruct the module before loading a
`state_dict()` into it.


In [ ]:
class FMRIEncoderMLP(nn.Module):
    def __init__(self, input_dim, output_dim=1280, dropout=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 2048),
            nn.LayerNorm(2048), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(2048, 4096),
            nn.LayerNorm(4096), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(4096, output_dim),
        )

    def forward(self, x):
        return self.net(x)


## Load saved models from Drive
Lists every run folder under `RUNS_DIR`, then loads the most recent `model.pth` found per loss type.
To evaluate specific runs instead of the latest, fill in `MODEL_PATHS` manually (e.g.
`MODEL_PATHS = {"mse": "/content/drive/MyDrive/braid2/runs/mlp_mse_20260819_153000"}`) before running
the cell below — auto-discovery only fills in entries you haven't already set.


In [ ]:
RUNS_DIR = "/content/drive/MyDrive/braid2/runs"

run_dirs = sorted(glob.glob(f"{RUNS_DIR}/mlp_*"), key=os.path.getmtime)
print(f"found {len(run_dirs)} run folder(s) under {RUNS_DIR}:")
for d in run_dirs:
    tag = "[model.pth]" if os.path.exists(f"{d}/model.pth") else "[no model] "
    print(f"  {tag}  {d}")


found 2 run folder(s) under /content/drive/MyDrive/braid2/runs:
  [model.pth]  /content/drive/MyDrive/braid2/runs/mlp_mse_20260820_040200
  [model.pth]  /content/drive/MyDrive/braid2/runs/mlp_cosine_20260820_040200


In [ ]:
# --- fill in explicit paths here to override auto-discovery for specific losses ---
MODEL_PATHS = {}   # e.g. {"mse": "/content/drive/MyDrive/braid2/runs/mlp_mse_20260819_153000"}

for d in run_dirs:                                    # oldest -> newest, so newest wins per loss
    if not os.path.exists(f"{d}/model.pth"):
        continue
    loss_type = os.path.basename(d).split("_")[1]      # "mlp_mse_20260819_153000" -> "mse"
    MODEL_PATHS.setdefault(loss_type, None)
    if MODEL_PATHS[loss_type] is None:
        MODEL_PATHS[loss_type] = d

print("evaluating:")
for lt, d in MODEL_PATHS.items():
    print(f"  {lt:10s} -> {d}")

LOSS_TYPES = list(MODEL_PATHS.keys())


evaluating:
  mse        -> /content/drive/MyDrive/braid2/runs/mlp_mse_20260820_040200
  cosine     -> /content/drive/MyDrive/braid2/runs/mlp_cosine_20260820_040200


In [ ]:
results = {}
for loss_type, run_dir in MODEL_PATHS.items():
    model = FMRIEncoderMLP(input_dim=betas.shape[1], output_dim=clips.shape[1]).to(DEVICE)
    model.load_state_dict(torch.load(f"{run_dir}/model.pth", map_location=DEVICE))
    model.eval()
    results[loss_type] = {"model": model, "run_dir": run_dir}
    print(f"[{loss_type}] loaded {run_dir}/model.pth")


[mse] loaded /content/drive/MyDrive/braid2/runs/mlp_mse_20260820_040200/model.pth
[cosine] loaded /content/drive/MyDrive/braid2/runs/mlp_cosine_20260820_040200/model.pth


## What to run
Two independent switches, since the two halves of this notebook have very different costs. Mixing
diagnostics only need the loaded models + test-set embeddings (seconds). Image evals need the full
shared-1000 test set decoded through Kandinsky first (tens of minutes per model) — skip it entirely if
you already have `all_recons.pt`/`eval_metrics.csv` saved from a previous pass and just want to re-run
mixing. The final table cell below merges freshly computed columns into whatever's already saved for
each run rather than overwriting it, so a mixing-only pass doesn't erase previously computed SSIM etc.


In [ ]:
RUN_MIXING      = True    # embedding-space mixing metrics + C2ST/MMD diagnostics -- no decode needed
RUN_IMAGE_EVALS = False   # PixCorr/SSIM/AlexNet/Inception/CLIP/EffNet/SwAV/Retrieval -- needs decoded images


## Decode the full shared-1000 test set (every loaded model)
Ground truth images are built once (loss-independent); each loaded model decodes all ~1000 test
images and caches the result to its run folder as `all_recons.pt`, so re-running the eval cells below
doesn't need to redo the (expensive) Kandinsky decode.


In [ ]:
if RUN_IMAGE_EVALS:
    !pip install -q diffusers accelerate
    import random, h5py
    from PIL import Image
    from diffusers import KandinskyV22Pipeline, KandinskyV22PriorPipeline

In [ ]:
if RUN_IMAGE_EVALS:
    STIM = f"{BASE}/nsd_stimuli.hdf5"

    # frozen Kandinsky decoder — conditions directly on a 1280-d CLIP image embed
    decoder = KandinskyV22Pipeline.from_pretrained(
        "kandinsky-community/kandinsky-2-2-decoder", torch_dtype=torch.float16).to(DEVICE)

    # proper NEGATIVE image embed (zero-image embed from the prior) — NOT literal zeros,
    # which cause a magenta color-cast under classifier-free guidance
    prior = KandinskyV22PriorPipeline.from_pretrained(
        "kandinsky-community/kandinsky-2-2-prior", torch_dtype=torch.float16).to(DEVICE)
    neg_embed = prior.get_zero_embed(1).to(DEVICE, torch.float16)      # [1, 1280]

    def ground_truth(img):           # imgBrick id -> PIL 512x512
        with h5py.File(STIM, "r") as f:
            return Image.fromarray(f["imgBrick"][int(img)]).resize((512, 512))

In [ ]:
if RUN_IMAGE_EVALS:
    print(f"building ground-truth image tensor for all {len(test_img_ids)} shared-1000 test images...")
    all_images = torch.stack([transforms.ToTensor()(ground_truth(i)) for i in tqdm(test_img_ids)])   # [N,3,512,512] in [0,1]
    print("all_images:", tuple(all_images.shape))

In [ ]:
if RUN_IMAGE_EVALS:
    @torch.no_grad()
    def decode_all(loss_type, batch_size=10):
        model   = results[loss_type]["model"]
        run_dir = results[loss_type]["run_dir"]

        cached = f"{run_dir}/all_recons.pt"
        if os.path.exists(cached):                                          # skip re-decoding if already done
            print(f"[{loss_type}] found cached {cached}, loading instead of re-decoding")
            return torch.load(cached)

        model.eval()
        xb  = ((test_betas - beta_mean) / beta_std).to(DEVICE)
        emb = model(xb) * clip_std.to(DEVICE) + clip_mean.to(DEVICE)
        if "cosine" in loss_type:                                           # cosine terms never constrain magnitude
            emb = emb / emb.norm(dim=1, keepdim=True) * clips[train_idx].norm(dim=1).mean()
        embeds = emb.half()

        recons = []
        for start in tqdm(range(0, len(embeds), batch_size), desc=f"[{loss_type}] decoding {len(embeds)} images"):
            batch = embeds[start:start + batch_size]
            imgs  = decoder(image_embeds=batch,
                            negative_image_embeds=neg_embed.repeat(len(batch), 1),
                            num_inference_steps=50, height=512, width=512).images
            recons.extend(transforms.ToTensor()(im) for im in imgs)

        all_recons = torch.stack(recons)                                    # [N,3,512,512] in [0,1]
        torch.save(all_recons, cached)
        print(f"[{loss_type}] saved: {cached}  {tuple(all_recons.shape)}")
        return all_recons


    for loss_type in LOSS_TYPES:
        results[loss_type]["all_recons"] = decode_all(loss_type)


## Evals
Image-reconstruction-quality metrics computed per loss over the full test set, following
[MindEye2's evals notebook](https://huggingface.co/datasets/pscotti/mindeyev2). Two categories from
that reference are **not** included here: brain-correlation via a pretrained GNet forward-encoding
model, and captioning metrics (METEOR/ROUGE/CLIP-text) — both need per-subject artifacts (a trained
image-to-voxel encoder, ROI masks, a brain-to-caption submodule) that this pipeline doesn't have and
that would need to be faked rather than ported. Everything below only needs pretrained, off-the-shelf
feature extractors plus the reconstructions/ground-truth images already in hand.


### Embedding metrics (MSE + cosine)
Direct probes in embedding space, not image space — no decode needed. For each positive pair
(predicted embedding, true embedding), computes MSE and cosine similarity, then averages over the
full test set. Runs for every loss regardless of which loss trained that model, so this is a fair,
loss-independent comparison in raw (de-normalized) CLIP space — the same space the Kandinsky decoder
actually consumes.


In [ ]:
@torch.no_grad()
def eval_embedding_metrics(loss_type):
    model = results[loss_type]["model"]
    model.eval()
    cm, cs = clip_mean.to(DEVICE), clip_std.to(DEVICE)

    P, T = [], []
    for x, y in test_loader:
        P.append((model(x.to(DEVICE)) * cs + cm).cpu())      # de-normalized, raw CLIP space
        T.append((y.to(DEVICE) * cs + cm).cpu())
    pred, true = torch.cat(P), torch.cat(T)

    mse = F.mse_loss(pred, true).item()                          # averaged over pairs and dims
    cos = F.cosine_similarity(pred, true, dim=1).mean().item()   # averaged over pairs
    print(f"[{loss_type}] embedding MSE: {mse:.4f}  |  embedding cosine similarity: {cos:.4f}")
    return mse, cos


### Mixing metrics — do predicted and true embeddings occupy the same region of space?
Quantitative replacement for eyeballing the PCA/UMAP plots, computed directly in the raw 1280-d
embedding space (not a lossy 2D projection). Pools predicted + true test embeddings and labels each
point by **source** (predicted vs. true), not any semantic category, then asks two versions of "can
you tell them apart":

- **Silhouette score** (cosine metric) on the source labels — repurposing a clustering-quality metric
  as a separability probe. Interpretation inverts from the usual reading: **~0 or negative = well
  mixed** (a predicted point is about as close to nearby true points as to other predicted points),
  **near +1 = separate clouds**. Same idea as "average silhouette width" (ASW) in the single-cell
  batch-effect-correction literature, where this exact question (do two point clouds occupy the same
  region, or are they systematically offset) is a standard, named evaluation.
- **Domain-classifier accuracy** — a 5-fold cross-validated logistic regression trained to predict
  source from the raw embedding. **0.50 = chance = well mixed**, **1.00 = trivially separable**.
  Catches non-convex separation that silhouette's local distance-based notion can miss.


In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

@torch.no_grad()
def mixing_metrics(loss_type):
    model = results[loss_type]["model"]
    model.eval()
    cm, cs = clip_mean.to(DEVICE), clip_std.to(DEVICE)

    P, T = [], []
    for x, y in test_loader:
        P.append((model(x.to(DEVICE)) * cs + cm).cpu())
        T.append((y.to(DEVICE) * cs + cm).cpu())
    pred, true = torch.cat(P).numpy(), torch.cat(T).numpy()

    X      = np.concatenate([pred, true])                      # [2N, 1280]
    source = np.array([0] * len(pred) + [1] * len(true))       # source label, NOT a semantic category

    silhouette = silhouette_score(X, source, metric="cosine")
    domain_acc = cross_val_score(LogisticRegression(max_iter=1000), X, source, cv=5).mean()

    print(f"[{loss_type}] mixing silhouette (cosine): {silhouette:.4f}  |  domain-classifier accuracy: {domain_acc:.4f}")
    return silhouette, domain_acc


### Mixing diagnostics — C2ST across geometric conditions, + MMD
Generalizes the domain-classifier check above along two axes: a second classifier (RBF SVM, which can
find a nonlinear separating boundary a linear logistic regression can't), and three geometric "views"
of the same embeddings, each stripping away a different kind of difference so the *specific* nature of
the gap can be localized rather than just detected:

- **Raw** — total geometric gap (magnitude + direction + shape, everything).
- **L2-normalized** — every vector rescaled to unit norm first, removing magnitude differences.
  Separability that *remains* here is direction/angle drift, not a scale issue.
  A cosine-loss model already ignores magnitude, so a big raw/L2-normalized gap for it points at scale.
- **Mean-centered** — each source's own centroid subtracted out first, removing any global offset
  between the two clouds. Separability that remains here is a *shape* difference (e.g. different
  effective dimensionality or spread), not just the clouds sitting in different places.

All three use the same classifier-two-sample-test (C2ST) logic as the domain-classifier metric above:
0.50 = indistinguishable under that view, 1.00 = trivially separable. **Maximum Mean Discrepancy**
(Gaussian/RBF kernel, median-heuristic bandwidth) is reported alongside as a non-classifier-based
distributional divergence — 0 would mean identical distributions; there's no fixed upper bound, so
read it relative to other losses/conditions rather than against an absolute scale.


In [ ]:
from sklearn.svm import SVC

def _l2_normalize(X):
    return X / np.linalg.norm(X, axis=1, keepdims=True)

def _mean_center(X):
    return X - X.mean(axis=0, keepdims=True)

def _pairwise_sq_dists(A, B):
    # ||a-b||^2 = ||a||^2 + ||b||^2 - 2 a.b, via matmul -- avoids an O(N*M*D) intermediate tensor
    A_sq = (A ** 2).sum(axis=1, keepdims=True)
    B_sq = (B ** 2).sum(axis=1, keepdims=True).T
    return np.clip(A_sq + B_sq - 2 * A @ B.T, a_min=0, a_max=None)

def mmd_gaussian(X, Y, gamma=None):
    """Squared MMD between X and Y with a Gaussian/RBF kernel (median-heuristic bandwidth if gamma unset)."""
    if gamma is None:
        Z   = np.concatenate([X, Y])
        sub = Z[np.random.choice(len(Z), size=min(len(Z), 500), replace=False)]
        d2  = _pairwise_sq_dists(sub, sub)
        gamma = 1.0 / (2 * np.median(d2[d2 > 0]))

    Kxx = np.exp(-gamma * _pairwise_sq_dists(X, X))
    Kyy = np.exp(-gamma * _pairwise_sq_dists(Y, Y))
    Kxy = np.exp(-gamma * _pairwise_sq_dists(X, Y))

    m, n = len(X), len(Y)
    mmd2 = ((Kxx.sum() - np.trace(Kxx)) / (m * (m - 1))
            + (Kyy.sum() - np.trace(Kyy)) / (n * (n - 1))
            - 2 * Kxy.mean())
    return float(mmd2)


@torch.no_grad()
def mixing_diagnostics(loss_type):
    model = results[loss_type]["model"]
    model.eval()
    cm, cs = clip_mean.to(DEVICE), clip_std.to(DEVICE)

    P, T = [], []
    for x, y in test_loader:
        P.append((model(x.to(DEVICE)) * cs + cm).cpu())
        T.append((y.to(DEVICE) * cs + cm).cpu())
    pred, true = torch.cat(P).numpy(), torch.cat(T).numpy()

    conditions = {
        "raw":           (pred, true),
        "l2_normalized": (_l2_normalize(pred), _l2_normalize(true)),
        "mean_centered": (_mean_center(pred), _mean_center(true)),
    }

    row = {}
    for cond_name, (p, t) in conditions.items():
        X = np.concatenate([p, t])
        y = np.array([0] * len(p) + [1] * len(t))

        logreg_acc = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=5).mean()
        svm_acc    = cross_val_score(SVC(kernel="rbf"), X, y, cv=5).mean()

        row[f"C2ST_LogReg_{cond_name}"] = logreg_acc
        row[f"C2ST_RBFSVM_{cond_name}"] = svm_acc
        print(f"[{loss_type}] C2ST ({cond_name:14s}) -> LogReg: {logreg_acc:.4f}  RBF-SVM: {svm_acc:.4f}")

    row["MMD"] = mmd_gaussian(pred, true)
    print(f"[{loss_type}] MMD (Gaussian kernel): {row['MMD']:.6f}")

    return row


### PixCorr
Raw pixel correlation between reconstruction and ground truth, per image, averaged.


In [ ]:
def eval_pixcorr(recons, images):
    resize = transforms.Resize(425, interpolation=transforms.InterpolationMode.BILINEAR)
    r = resize(images).reshape(len(images), -1).cpu().numpy()
    f = resize(recons).reshape(len(recons), -1).cpu().numpy()
    return float(np.mean([np.corrcoef(r[i], f[i])[0, 1] for i in range(len(r))]))


### SSIM
Structural similarity, computed on grayscale images (see MindEye2 / meshconv-decoding issue #3 for why
grayscale is standard here).


In [ ]:
from skimage.color import rgb2gray
from skimage.metrics import structural_similarity as ssim_fn

def eval_ssim(recons, images):
    resize   = transforms.Resize(425, interpolation=transforms.InterpolationMode.BILINEAR)
    img_gray = rgb2gray(resize(images).permute(0, 2, 3, 1).cpu().numpy())
    rec_gray = rgb2gray(resize(recons).permute(0, 2, 3, 1).cpu().numpy())
    scores = [ssim_fn(rec, im, data_range=1.0, gaussian_weights=True, sigma=1.5, use_sample_covariance=False)
              for rec, im in zip(rec_gray, img_gray)]
    return float(np.mean(scores))


### Two-way identification
Shared helper for the next four metrics: embed every reconstruction and every ground-truth image
through a frozen pretrained feature extractor, then for each image check whether its reconstruction's
features correlate with it more strongly than with every *other* ground-truth image (a pairwise
forced-choice test). Returns the fraction of correct forced choices, averaged over all comparisons.


In [ ]:
@torch.no_grad()
def two_way_identification(recons, images, model, preprocess, feature_layer=None):
    preds = model(torch.stack([preprocess(r) for r in recons]).to(DEVICE))
    reals = model(torch.stack([preprocess(im) for im in images]).to(DEVICE))
    if feature_layer is not None:
        preds, reals = preds[feature_layer], reals[feature_layer]
    preds = preds.float().flatten(1).cpu().numpy()
    reals = reals.float().flatten(1).cpu().numpy()

    r = np.corrcoef(reals, preds)
    r = r[:len(images), len(images):]
    congruents = np.diag(r)
    success = r < congruents
    return float(np.mean(np.sum(success, 0)) / (len(images) - 1))


### AlexNet (early + mid layers)


In [ ]:
if RUN_IMAGE_EVALS:
    from torchvision.models.feature_extraction import create_feature_extractor
    from torchvision.models import alexnet, AlexNet_Weights

    alex_model = create_feature_extractor(
        alexnet(weights=AlexNet_Weights.IMAGENET1K_V1), return_nodes=["features.4", "features.11"]
    ).to(DEVICE).eval().requires_grad_(False)

    alex_preprocess = transforms.Compose([
        transforms.Resize(256, interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

### InceptionV3


In [ ]:
if RUN_IMAGE_EVALS:
    from torchvision.models import inception_v3, Inception_V3_Weights

    inception_model = create_feature_extractor(
        inception_v3(weights=Inception_V3_Weights.DEFAULT), return_nodes=["avgpool"]
    ).to(DEVICE).eval().requires_grad_(False)

    inception_preprocess = transforms.Compose([
        transforms.Resize(342, interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

### CLIP
Uses OpenAI's `clip` package directly (final-layer image embedding) for the two-way identification
test — separate from the Kandinsky-paired CLIP-ViT-bigG-14 embeddings the encoder itself regresses
into; this is purely a perceptual-similarity backbone for scoring reconstruction quality.


In [ ]:
if RUN_IMAGE_EVALS:
    !pip install -q git+https://github.com/openai/CLIP.git
    import clip as openai_clip

    clip_2way_model, _ = openai_clip.load("ViT-L/14", device=DEVICE)

    clip_2way_preprocess = transforms.Compose([
        transforms.Resize(224, interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711]),
    ])

### EfficientNet-B1
Unlike the metrics above, EffNet and SwAV are scored with feature-space **correlation distance**
(lower = more similar) rather than two-way identification, matching the reference notebook.


In [ ]:
if RUN_IMAGE_EVALS:
    import scipy as sp
    from torchvision.models import efficientnet_b1, EfficientNet_B1_Weights

    eff_model = create_feature_extractor(
        efficientnet_b1(weights=EfficientNet_B1_Weights.DEFAULT), return_nodes=["avgpool"]
    ).to(DEVICE).eval().requires_grad_(False)

    eff_preprocess = transforms.Compose([
        transforms.Resize(255, interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    @torch.no_grad()
    def eval_effnet(recons, images):
        gt = eff_model(eff_preprocess(images).to(DEVICE))["avgpool"].reshape(len(images), -1).cpu().numpy()
        fk = eff_model(eff_preprocess(recons).to(DEVICE))["avgpool"].reshape(len(recons), -1).cpu().numpy()
        return float(np.mean([sp.spatial.distance.correlation(gt[i], fk[i]) for i in range(len(gt))]))

### SwAV


In [ ]:
if RUN_IMAGE_EVALS:
    swav_model = torch.hub.load("facebookresearch/swav:main", "resnet50")
    swav_model = create_feature_extractor(swav_model, return_nodes=["avgpool"]).to(DEVICE).eval().requires_grad_(False)

    swav_preprocess = transforms.Compose([
        transforms.Resize(224, interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    @torch.no_grad()
    def eval_swav(recons, images):
        gt = swav_model(swav_preprocess(images).to(DEVICE))["avgpool"].reshape(len(images), -1).cpu().numpy()
        fk = swav_model(swav_preprocess(recons).to(DEVICE))["avgpool"].reshape(len(recons), -1).cpu().numpy()
        return float(np.mean([sp.spatial.distance.correlation(gt[i], fk[i]) for i in range(len(gt))]))

### Retrieval (forward / backward top-1 identification)
Reuses the CLIP embeddings already computed during training/eval (`test_clips` = true, the encoder's
own predictions = brain-derived) rather than loading a separate embedding model — this pipeline only
ever has one embedding space, unlike MindEye2's retrieval submodule. 30 random 300-image draws, top-1
accuracy each way, with a 95% CI, matching the reference notebook's protocol.


In [ ]:
from scipy import stats

def batchwise_cosine_similarity(A, B):
    A = F.normalize(A, dim=-1)
    B = F.normalize(B, dim=-1)
    return A @ B.T

def topk_acc(sim, labels, k=1):
    _, top_idx = sim.topk(k, dim=1)
    return (top_idx == labels.unsqueeze(1)).any(dim=1).float().mean().item()

@torch.no_grad()
def eval_retrieval(loss_type, n_loops=30, n_samples=300):
    model = results[loss_type]["model"]
    model.eval()
    cm, cs = clip_mean.to(DEVICE), clip_std.to(DEVICE)

    xb          = ((test_betas - beta_mean) / beta_std).to(DEVICE)
    pred_embeds = (model(xb) * cs + cm).float()          # [N_test, 1280], raw CLIP space
    true_embeds = test_clips.to(DEVICE).float()          # [N_test, 1280]

    n = len(true_embeds)
    fwd, bwd = [], []
    for _ in range(n_loops):
        idx    = np.random.choice(n, size=min(n_samples, n), replace=False)
        labels = torch.arange(len(idx)).to(DEVICE)
        fwd_sim = batchwise_cosine_similarity(pred_embeds[idx], true_embeds[idx])   # brain -> image
        bwd_sim = batchwise_cosine_similarity(true_embeds[idx], pred_embeds[idx])   # image -> brain
        fwd.append(topk_acc(fwd_sim, labels, k=1))
        bwd.append(topk_acc(bwd_sim, labels, k=1))

    fwd_mean, bwd_mean = float(np.mean(fwd)), float(np.mean(bwd))
    fwd_ci = stats.norm.interval(0.95, loc=fwd_mean, scale=np.std(fwd) / np.sqrt(n_loops))
    bwd_ci = stats.norm.interval(0.95, loc=bwd_mean, scale=np.std(bwd) / np.sqrt(n_loops))
    print(f"[{loss_type}] fwd retrieval: {fwd_mean:.4f}  95% CI [{fwd_ci[0]:.4f},{fwd_ci[1]:.4f}]")
    print(f"[{loss_type}] bwd retrieval: {bwd_mean:.4f}  95% CI [{bwd_ci[0]:.4f},{bwd_ci[1]:.4f}]")
    return fwd_mean, bwd_mean


### Run all evals + comparison table


In [ ]:
import pandas as pd

eval_rows = []
for loss_type in LOSS_TYPES:
    run_dir = results[loss_type]["run_dir"]
    print(f"\n=== evals: loss = {loss_type} ===")

    # start from whatever's already saved for this run, so a partial (e.g. mixing-only) pass
    # doesn't erase columns computed in an earlier full pass
    existing_path = f"{run_dir}/eval_metrics.csv"
    if os.path.exists(existing_path):
        row = pd.read_csv(existing_path).iloc[0].to_dict()
        row["loss"] = loss_type
        print(f"[{loss_type}] loaded existing {existing_path}, updating in place")
    else:
        row = {"loss": loss_type}

    if RUN_MIXING:
        embed_mse, embed_cos          = eval_embedding_metrics(loss_type)
        mix_silhouette, mix_domain_acc = mixing_metrics(loss_type)
        diag_row                      = mixing_diagnostics(loss_type)
        row.update({
            "EmbedMSE":      embed_mse,
            "EmbedCosine":   embed_cos,
            "MixSilhouette": mix_silhouette,
            "MixDomainAcc":  mix_domain_acc,
        })
        row.update(diag_row)

    if RUN_IMAGE_EVALS:
        recons = results[loss_type]["all_recons"]
        row.update({
            "PixCorr":      eval_pixcorr(recons, all_images),
            "SSIM":         eval_ssim(recons, all_images),
            "AlexNet(2)":   two_way_identification(recons, all_images, alex_model, alex_preprocess, "features.4"),
            "AlexNet(5)":   two_way_identification(recons, all_images, alex_model, alex_preprocess, "features.11"),
            "InceptionV3":  two_way_identification(recons, all_images, inception_model, inception_preprocess, "avgpool"),
            "CLIP":         two_way_identification(recons, all_images, clip_2way_model.encode_image, clip_2way_preprocess, None),
            "EffNet-B":     eval_effnet(recons, all_images),
            "SwAV":         eval_swav(recons, all_images),
        })
        row["FwdRetrieval"], row["BwdRetrieval"] = eval_retrieval(loss_type)

    eval_rows.append(row)
    print(row)

    pd.DataFrame([row]).set_index("loss").to_csv(existing_path)
    print(f"[{loss_type}] saved: {existing_path}")

eval_df = pd.DataFrame(eval_rows).set_index("loss")
print("\n=== eval summary (all losses) ===")
print(eval_df.to_string())

EVAL_TAG = time.strftime('%Y%m%d_%H%M%S')
eval_df.to_csv(f"{RUNS_DIR}/eval_summary_evalrun_{EVAL_TAG}.csv")
print(f"\nsaved: {RUNS_DIR}/eval_summary_evalrun_{EVAL_TAG}.csv")



=== evals: loss = mse ===
[mse] loaded existing /content/drive/MyDrive/braid2/runs/mlp_mse_20260820_040200/eval_metrics.csv, updating in place
[mse] embedding MSE: 0.8071  |  embedding cosine similarity: 0.6412
[mse] mixing silhouette (cosine): 0.1292  |  domain-classifier accuracy: 0.8260
[mse] C2ST (raw           ) -> LogReg: 0.8260  RBF-SVM: 1.0000
[mse] C2ST (l2_normalized ) -> LogReg: 0.9990  RBF-SVM: 1.0000
[mse] C2ST (mean_centered ) -> LogReg: 0.4755  RBF-SVM: 1.0000
[mse] MMD (Gaussian kernel): 0.081132
{'loss': 'mse', 'EmbedMSE': 0.8070860505104065, 'EmbedCosine': 0.6411634087562561, 'MixSilhouette': np.float32(0.1292097), 'MixDomainAcc': np.float64(0.826), 'PixCorr': 0.158333348886868, 'SSIM': 0.2980650921307564, 'AlexNet(2)': 0.8490680680680681, 'AlexNet(5)': 0.8982612612612613, 'InceptionV3': 0.8145405405405406, 'CLIP': 0.8333283283283284, 'EffNet-B': 0.8158155083656311, 'SwAV': 0.4536454081535339, 'FwdRetrieval': 0.0665555572758118, 'BwdRetrieval': 0.1488888936738173, 'C

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
